# Static occupation against a nearby CORS, RTK baseline edition

This notebook uses one IGS station as the occupation and one nearby IGS station as the known reference. The executed path uses local core fixtures when `SIDEREON_CORE_FIXTURES` is set, and otherwise downloads the same public source files into a scratch cache.

Occupation: WTZZ, treated as the fast-static rover log.

Reference station: WTZR, held fixed at its published ITRF marker coordinate plus the RINEX antenna height.

Public source files:

- WTZR observation RINEX: https://epncb.oma.be/pub/RINEX/2020/177/WTZR00DEU_R_20201770000_01D_30S_MO.crx.gz
- WTZZ observation RINEX: https://epncb.oma.be/pub/RINEX/2020/177/WTZZ00DEU_R_20201770000_01D_30S_MO.crx.gz
- CODE MGEX precise orbit: https://igs.bkg.bund.de/root_ftp/IGS/products/mgex/2111/com21114.eph.Z

The solver path is deliberately thin: build RTK arcs from raw RINEX through the Python binding, solve a static float baseline, then solve the dual-frequency wide-lane fixed baseline. The table compares the corrected WTZZ antenna reference point against the published ITRF coordinate with the RINEX antenna height applied.


In [1]:
import gzip
import json
import math
import os
import subprocess
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import sidereon

OCCUPATION_STATION = "WTZZ"
REFERENCE_STATION = "WTZR"
ARC_EPOCH_LIMIT = 120
BOUND_SUBARCS = 4

PUBLIC_URLS = {
    "reference_obs": "https://epncb.oma.be/pub/RINEX/2020/177/WTZR00DEU_R_20201770000_01D_30S_MO.crx.gz",
    "occupation_obs": "https://epncb.oma.be/pub/RINEX/2020/177/WTZZ00DEU_R_20201770000_01D_30S_MO.crx.gz",
    "sp3": "https://igs.bkg.bund.de/root_ftp/IGS/products/mgex/2111/com21114.eph.Z",
}

FIXTURE_NAMES = {
    "reference_obs": ("obs", "WTZR00DEU_R_20201770000_01D_30S_MO_120epoch.rnx"),
    "occupation_obs": ("obs", "WTZZ00DEU_R_20201770000_01D_30S_MO_120epoch.rnx"),
    "sp3": ("sp3", "GBM0MGXRAP_20201770000_01D_05M_ORB_120epoch.sp3"),
}

PUBLISHED_MARKER_M = {
    "WTZR": np.array([4075580.3111, 931854.0543, 4801568.2808], dtype=float),
    "WTZZ": np.array([4075579.1913, 931853.3696, 4801569.1897], dtype=float),
}


def local_name(url):
    parsed = urllib.parse.urlparse(url)
    return Path(parsed.path).name


def download(url, data_dir):
    data_dir.mkdir(parents=True, exist_ok=True)
    path = data_dir / local_name(url)
    if not path.exists() or path.stat().st_size == 0:
        urllib.request.urlretrieve(url, path)
    return path


def gzip_text(path):
    return gzip.open(path, "rb").read().decode("utf-8", "replace")


def unix_uncompress_bytes(path):
    return subprocess.check_output(["gzip", "-cd", str(path)])


def first_rinex3_epochs(text, epoch_count):
    lines = text.splitlines(True)
    out = []
    idx = 0
    while idx < len(lines):
        out.append(lines[idx])
        if "END OF HEADER" in lines[idx]:
            idx += 1
            break
        idx += 1

    copied = 0
    while idx < len(lines) and copied < epoch_count:
        line = lines[idx]
        if line.startswith(">"):
            sat_count = int(line.split()[8])
            out.append(line)
            idx += 1
            for _ in range(sat_count):
                out.append(lines[idx])
                idx += 1
            copied += 1
        else:
            idx += 1
    return "".join(out)


def fixture_paths():
    fixture_root = os.environ.get("SIDEREON_CORE_FIXTURES")
    if not fixture_root:
        return None
    root = Path(fixture_root)
    paths = {key: root / parts[0] / parts[1] for key, parts in FIXTURE_NAMES.items()}
    return paths if all(path.exists() for path in paths.values()) else None


def load_public_data():
    data_dir = Path(
        os.environ.get("SIDEREON_CORS_DATA", "/private/tmp/sidereon-cors-data")
    )
    paths = {key: download(url, data_dir) for key, url in PUBLIC_URLS.items()}
    reference_text = sidereon.decode_crinex(gzip_text(paths["reference_obs"]))
    occupation_text = sidereon.decode_crinex(gzip_text(paths["occupation_obs"]))
    reference_obs = sidereon.parse_rinex_obs(
        first_rinex3_epochs(reference_text, ARC_EPOCH_LIMIT)
    )
    occupation_obs = sidereon.parse_rinex_obs(
        first_rinex3_epochs(occupation_text, ARC_EPOCH_LIMIT)
    )
    sp3 = sidereon.load_sp3(unix_uncompress_bytes(paths["sp3"]))
    return sp3, reference_obs, occupation_obs, "public downloads"


def load_data():
    paths = fixture_paths()
    if paths is None:
        return load_public_data()
    sp3 = sidereon.load_sp3(paths["sp3"])
    reference_obs = sidereon.load_rinex_obs(paths["reference_obs"])
    occupation_obs = sidereon.load_rinex_obs(paths["occupation_obs"])
    return sp3, reference_obs, occupation_obs, "local core fixtures"


sp3, reference_obs, occupation_obs, DATA_SOURCE = load_data()
print(f"Data source: {DATA_SOURCE}")
print(f"Reference station: {REFERENCE_STATION}")
print(f"Occupation station: {OCCUPATION_STATION}")

Data source: local core fixtures
Reference station: WTZR
Occupation station: WTZZ


## QC First

Before solving the baseline, inspect the occupation log. The QC pass looks for gaps, cycle-slip flags, and carrier-code multipath levels by constellation. This is the notebook equivalent of checking whether a field log deserves a static solve.


In [2]:
qc = json.loads(sidereon.observation_qc(occupation_obs).to_json())

print(f"Observation epochs: {qc['observation_epochs']}")
print(f"Missing epochs: {qc['missing_epochs']}")
print(f"Cycle-slip flags: {qc['cycle_slips']['total_slips']}")
print(f"Observations per slip flag: {qc['cycle_slips']['observations_per_slip']:.1f}")
print()
print("| system | MP1 RMS m | MP2 RMS m | slip flags |")
print("|---|---:|---:|---:|")
slips = {row["system"]: row["slips"] for row in qc["cycle_slips"]["by_system"]}
for row in qc["multipath"]["systems"]:
    system = row["system"]
    print(
        f"| {system} | {row['mp1']['rms_m']:.3f} | "
        f"{row['mp2']['rms_m']:.3f} | {slips.get(system, 0)} |"
    )

Observation epochs: 120
Missing epochs: 0
Cycle-slip flags: 79
Observations per slip flag: 54.7

| system | MP1 RMS m | MP2 RMS m | slip flags |
|---|---:|---:|---:|
| Gps | 0.249 | 0.236 | 9 |
| Glonass | 0.353 | 0.349 | 17 |
| Galileo | 0.279 | 0.161 | 13 |
| BeiDou | 0.539 | 0.191 | 40 |


## Published Truth

The comparison is made at the WTZZ antenna reference point because the RTK baseline is solved between antenna reference points. The published ITRF marker coordinate is adjusted by the RINEX antenna height along the local up direction before comparing the corrected coordinate.


In [3]:
def ecef_to_geodetic(xyz):
    x, y, z = map(float, xyz)
    semi_major_m = 6_378_137.0
    flattening = 1 / 298.257223563
    eccentricity_sq = flattening * (2 - flattening)
    lon = math.atan2(y, x)
    p = math.hypot(x, y)
    lat = math.atan2(z, p * (1 - eccentricity_sq))
    for _ in range(8):
        sin_lat = math.sin(lat)
        prime_vertical = semi_major_m / math.sqrt(1 - eccentricity_sq * sin_lat**2)
        height = p / math.cos(lat) - prime_vertical
        lat = math.atan2(
            z,
            p * (1 - eccentricity_sq * prime_vertical / (prime_vertical + height)),
        )
    sin_lat = math.sin(lat)
    prime_vertical = semi_major_m / math.sqrt(1 - eccentricity_sq * sin_lat**2)
    height = p / math.cos(lat) - prime_vertical
    return lat, lon, height


def enu_matrix(reference_xyz):
    lat, lon, _ = ecef_to_geodetic(reference_xyz)
    sin_lat, cos_lat = math.sin(lat), math.cos(lat)
    sin_lon, cos_lon = math.sin(lon), math.cos(lon)
    return np.array(
        [
            [-sin_lon, cos_lon, 0.0],
            [-sin_lat * cos_lon, -sin_lat * sin_lon, cos_lat],
            [cos_lat * cos_lon, cos_lat * sin_lon, sin_lat],
        ]
    )


def antenna_reference_point(marker_xyz, obs):
    height_m, east_m, north_m = obs.header.antenna_delta_hen_m
    if abs(east_m) > 1e-12 or abs(north_m) > 1e-12:
        raise ValueError("this notebook expects zero east and north antenna offsets")
    return marker_xyz + enu_matrix(marker_xyz)[2] * height_m


def enu_error(position_xyz, truth_xyz):
    return enu_matrix(truth_xyz) @ (np.asarray(position_xyz, dtype=float) - truth_xyz)


def horizontal_vertical_error(position_xyz, truth_xyz):
    east_m, north_m, up_m = enu_error(position_xyz, truth_xyz)
    return float(math.hypot(east_m, north_m)), float(up_m)


reference_marker_m = PUBLISHED_MARKER_M[REFERENCE_STATION]
occupation_marker_m = PUBLISHED_MARKER_M[OCCUPATION_STATION]
reference_arp_m = antenna_reference_point(reference_marker_m, reference_obs)
occupation_truth_arp_m = antenna_reference_point(occupation_marker_m, occupation_obs)
truth_baseline_m = occupation_truth_arp_m - reference_arp_m

print("WTZR published marker ECEF m:", np.round(reference_marker_m, 4).tolist())
print("WTZZ published marker ECEF m:", np.round(occupation_marker_m, 4).tolist())
print("Truth rover-minus-base ARP baseline m:", np.round(truth_baseline_m, 4).tolist())

WTZR published marker ECEF m: [4075580.3111, 931854.0543, 4801568.2808]
WTZZ published marker ECEF m: [4075579.1913, 931853.3696, 4801569.1897]
Truth rover-minus-base ARP baseline m: [-0.984, -0.6536, 1.07]


## RTK Baseline Solves

The single-frequency path builds raw RTK epochs from RINEX and solves a static float baseline. The dual-frequency path builds a dual-frequency RINEX arc, fixes wide-lane ambiguities, prepares the ionosphere-free arc, and solves the static fixed baseline.


In [4]:
def rtk_model():
    return sidereon.RtkMeasurementModel(
        code_sigma_m=2.0,
        phase_sigma_m=0.01,
        sagnac=True,
        stochastic=sidereon.RtkStochasticModel.SIMPLE,
        elevation_weighting=True,
    )


def float_options():
    return sidereon.RtkFloatOptions(
        position_tol_m=1.0e-4,
        ambiguity_tol_m=1.0e-4,
        max_iterations=10,
    )


def fixed_options():
    return sidereon.RtkFixedOptions(
        position_tol_m=1.0e-4,
        ambiguity_tol_m=1.0e-4,
        max_iterations=10,
        ratio_threshold=3.0,
        partial_ambiguity_resolution=False,
        partial_min_ambiguities=4,
    )


model = rtk_model()
float_opts = float_options()
fixed_opts = fixed_options()
single_arc_options = sidereon.RtkRinexArcOptions(
    max_epochs=ARC_EPOCH_LIMIT,
    include_prediction_time=False,
)
dual_arc_options = sidereon.RtkRinexDualArcOptions(
    max_epochs=ARC_EPOCH_LIMIT,
    include_prediction_time=False,
)

single_arc = sidereon.build_rinex_rtk_arc(
    sp3,
    reference_obs,
    occupation_obs,
    single_arc_options,
)
dual_arc = sidereon.build_dual_frequency_rinex_rtk_arc(
    sp3,
    reference_obs,
    occupation_obs,
    dual_arc_options,
)

single_solution = sidereon.solve_static_rinex_rtk_baseline(
    sp3,
    reference_obs,
    occupation_obs,
    reference_arp_m.tolist(),
    model=model,
    arc_options=single_arc_options,
    preprocessing=sidereon.RtkArcPreprocessing(cycle_slip="split_arc"),
    float_options=float_opts,
    fixed_options=fixed_opts,
)
wide_solution = sidereon.solve_wide_lane_fixed_rinex_rtk_baseline(
    sp3,
    reference_obs,
    occupation_obs,
    reference_arp_m.tolist(),
    model=model,
    arc_options=dual_arc_options,
    float_options=float_opts,
    fixed_options=fixed_opts,
)

print(f"Single-frequency RTK epochs: {len(single_arc.epochs)}")
print(f"Dual-frequency RTK epochs: {len(dual_arc.epochs)}")
print(f"Single-frequency reference selection: {dict(single_solution.references)}")
print(f"Wide-lane fixed: {wide_solution.wide_lane_fixed}")
print(f"Wide-lane integer status: {wide_solution.integer_status!r}")
print(f"Wide-lane integer ratio: {wide_solution.integer_ratio:.3f}")
print(f"Single-frequency split arcs: {len(single_solution.split_cycle_slip_arcs)}")

Single-frequency RTK epochs: 120
Dual-frequency RTK epochs: 120
Single-frequency reference selection: {'G': 'G30'}
Wide-lane fixed: True
Wide-lane integer status: IntegerStatus.FIXED
Wide-lane integer ratio: 17.324
Single-frequency split arcs: 4


## Empirical Check Bounds

The RTK static baseline result does not expose a coordinate covariance. The bounds below are therefore validation bounds: solve equal sub-arcs, compare those corrected coordinates to the same published truth, and report a two-rms truth-centered check bound. For fixed wide-lane bounds, only sub-arcs whose integer search fixed are included.


In [5]:
def split_equal(items, parts):
    total = len(items)
    for idx in range(parts):
        start = round(idx * total / parts)
        stop = round((idx + 1) * total / parts)
        if stop > start:
            yield items[start:stop]


def static_arc_config(wavelengths_m, offsets_m, preprocessing=None, references=None):
    return sidereon.RtkStaticArcConfig(
        sidereon.RtkArcConfig(
            base=reference_arp_m.tolist(),
            model=model,
            wavelengths_m=wavelengths_m,
            offsets_m=offsets_m,
            baseline_prior_sigma_m=30.0,
            ambiguity_prior_sigma_m=30.0,
            initial_baseline_m=[0.0, 0.0, 0.0],
            preprocessing=preprocessing,
            reference_per_system=references,
        ),
        float_options=float_opts,
        fixed_options=fixed_opts,
    )


def corrected_position(baseline_m):
    return reference_arp_m + np.asarray(baseline_m, dtype=float)


def solve_float_subarc(chunk):
    solution = sidereon.solve_static_rtk_arc(
        chunk,
        static_arc_config(
            single_arc.wavelengths_m,
            single_arc.offsets_m,
            preprocessing=sidereon.RtkArcPreprocessing(cycle_slip="split_arc"),
        ),
    )
    return corrected_position(solution.float_solution.baseline_m)


def solve_wide_lane_subarc(chunk):
    wide_lane = sidereon.fix_wide_lane_rtk_arc(
        chunk,
        sidereon.RtkWideLaneArcConfig(
            base=reference_arp_m.tolist(),
            options=sidereon.RtkWideLaneOptions(
                min_epochs=2,
                tolerance_cycles=0.5,
                skip_short_fragments=False,
            ),
            cycle_slip=sidereon.RtkDualCycleSlipConfig("drop_satellite"),
        ),
    )
    ionosphere_free = sidereon.prepare_ionosphere_free_rtk_arc(
        chunk,
        wide_lane.wide_lane_cycles,
        sidereon.RtkIonosphereFreeArcConfig(
            base=reference_arp_m.tolist(),
            reference_per_system=wide_lane.references,
            apply_troposphere=True,
        ),
    )
    solution = sidereon.solve_static_rtk_arc(
        ionosphere_free.epochs,
        static_arc_config(
            ionosphere_free.wavelengths_m,
            ionosphere_free.offsets_m,
            references=ionosphere_free.references,
        ),
    )
    return solution.fixed_solution, corrected_position(
        solution.fixed_solution.fixed_baseline_m
    )


def check_bound(positions, full_position):
    if not positions:
        return float("nan"), float("nan")
    enu = np.array(
        [enu_error(position, occupation_truth_arp_m) for position in positions]
    )
    horizontal = np.hypot(enu[:, 0], enu[:, 1])
    full_horizontal, full_vertical = horizontal_vertical_error(
        full_position,
        occupation_truth_arp_m,
    )
    horizontal_bound = 2.0 * math.sqrt(float(np.mean(horizontal**2)))
    vertical_bound = 2.0 * math.sqrt(float(np.mean(enu[:, 2] ** 2)))
    return (
        max(abs(full_horizontal), horizontal_bound),
        max(abs(full_vertical), vertical_bound),
    )


float_subarc_positions = [
    solve_float_subarc(chunk) for chunk in split_equal(single_arc.epochs, BOUND_SUBARCS)
]

wide_subarc_positions = []
wide_subarc_statuses = []
for chunk in split_equal(dual_arc.epochs, BOUND_SUBARCS):
    fixed_solution, position = solve_wide_lane_subarc(chunk)
    wide_subarc_statuses.append(
        (repr(fixed_solution.integer_status), fixed_solution.integer_ratio)
    )
    if fixed_solution.integer_status == sidereon.IntegerStatus.FIXED:
        wide_subarc_positions.append(position)

float_position = corrected_position(single_solution.float_solution.baseline_m)
wide_position = corrected_position(wide_solution.fixed_baseline_m)
float_bound_h_m, float_bound_v_m = check_bound(float_subarc_positions, float_position)
wide_bound_h_m, wide_bound_v_m = check_bound(wide_subarc_positions, wide_position)

print(f"Float sub-arcs used for bounds: {len(float_subarc_positions)}")
print(f"Wide-lane fixed sub-arcs used for bounds: {len(wide_subarc_positions)}")
print("Wide-lane sub-arc statuses:")
for status, ratio in wide_subarc_statuses:
    print(f"  {status}, ratio {ratio:.3f}")

Float sub-arcs used for bounds: 4
Wide-lane fixed sub-arcs used for bounds: 3
Wide-lane sub-arc statuses:
  IntegerStatus.FIXED, ratio 6.213
  IntegerStatus.FIXED, ratio 4.049
  IntegerStatus.FIXED, ratio 6.498
  IntegerStatus.NOT_FIXED, ratio 1.260


## Honest Table

The table reports the corrected WTZZ antenna reference point against the published ITRF truth. The fixed single-frequency candidate is intentionally not used as the final fixed result because its integer search did not accept a fix on this arc. The wide-lane fixed line is the accepted dual-frequency fixed result.


In [6]:
def solution_row(
    method, position, horizontal_bound_m, vertical_bound_m, status, chunks_used
):
    east_m, north_m, up_m = enu_error(position, occupation_truth_arp_m)
    horizontal_m = math.hypot(east_m, north_m)
    return {
        "method": method,
        "east_cm": east_m * 100.0,
        "north_cm": north_m * 100.0,
        "up_cm": up_m * 100.0,
        "horizontal_cm": horizontal_m * 100.0,
        "vertical_cm": up_m * 100.0,
        "horizontal_bound_cm": horizontal_bound_m * 100.0,
        "vertical_bound_cm": vertical_bound_m * 100.0,
        "status": status,
        "chunks_used": chunks_used,
    }


summary_rows = [
    solution_row(
        "Single-frequency static RTK float",
        float_position,
        float_bound_h_m,
        float_bound_v_m,
        "float",
        len(float_subarc_positions),
    ),
    solution_row(
        "Dual-frequency wide-lane fixed RTK",
        wide_position,
        wide_bound_h_m,
        wide_bound_v_m,
        repr(wide_solution.integer_status),
        len(wide_subarc_positions),
    ),
]

print("Corrected WTZZ ARP ECEF m:")
print("  float:", np.round(float_position, 4).tolist())
print("  wide-lane fixed:", np.round(wide_position, 4).tolist())
print("Published WTZZ ARP truth ECEF m:", np.round(occupation_truth_arp_m, 4).tolist())
print()
print(
    "| method | east cm | north cm | up cm | horizontal cm | "
    "vertical cm | horizontal bound cm | vertical bound cm | status | bound chunks |"
)
print("|---|---:|---:|---:|---:|---:|---:|---:|---|---:|")
for row in summary_rows:
    print(
        f"| {row['method']} | {row['east_cm']:.3f} | {row['north_cm']:.3f} | "
        f"{row['up_cm']:.3f} | {row['horizontal_cm']:.3f} | "
        f"{row['vertical_cm']:.3f} | {row['horizontal_bound_cm']:.3f} | "
        f"{row['vertical_bound_cm']:.3f} | {row['status']} | {row['chunks_used']} |"
    )

Corrected WTZZ ARP ECEF m:
  float: [4075579.3703, 931853.4034, 4801569.4028]
  wide-lane fixed: [4075579.3745, 931853.4141, 4801569.3969]
Published WTZZ ARP truth ECEF m: [4075579.3724, 931853.411, 4801569.4045]

| method | east cm | north cm | up cm | horizontal cm | vertical cm | horizontal bound cm | vertical bound cm | status | bound chunks |
|---|---:|---:|---:|---:|---:|---:|---:|---|---:|
| Single-frequency static RTK float | -0.699 | 0.176 | -0.377 | 0.721 | -0.377 | 8.414 | 3.916 | float | 4 |
| Dual-frequency wide-lane fixed RTK | 0.259 | -0.703 | -0.393 | 0.749 | -0.393 | 0.995 | 1.175 | IntegerStatus.FIXED | 3 |


## Static-Solve Binding Audit

`sidereon-core` exposes a multi-epoch static pseudorange solve with covariance, and this repository already binds it as `solve_static`. For reference-station static RTK, core exposes static arc and RINEX baseline solves, and those are already bound here, but the public core result does not expose a baseline coordinate covariance. This notebook therefore reports validation bounds from repeated sub-arcs instead of adding a Python-side covariance surrogate.
